<a href="https://colab.research.google.com/github/Fillo468/Emotion-Recognition-Multimodal-Fusion-/blob/main/REGRESSION_AMIGOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Dataset**

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from google.colab import drive
drive.mount('/content/drive')

# Uploading the preprocessed dataset
path = "/content/drive/MyDrive/amigos_features.csv"
df = pd.read_csv(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Regression Computation**

In [24]:
# Selecting only the physiological features
ecg_features = [c for c in df.columns if c.startswith('ecg_') and c != 'ecg_quality']
eda_features = [c for c in df.columns if c.startswith('eda_') and c != 'eda_quality']

# Setting targets
y_arousal = df['arousal']
y_valence = df['valence']

In [25]:
# Function to train & evaluate accuracy metrics of each single regressor independently
def train_and_evaluate(X, y, params):

    # Splitting the dataset into train, validation & test sets
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

    # Normalization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # Searching for best parameters for each single regressor
    # rf = RandomForestRegressor(random_state=42)
    # grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5...)
    # grid_search.fit(X_train_scaled, y_train)
    # best_model = grid_search.best_estimator_

    # Creating Regressor
    best_model = RandomForestRegressor(random_state=42, **params)

    # Training Regressor
    best_model.fit(X_train_scaled, y_train)

    # Computing regression & evaluating accuracy metrics on validation set
    y_pred_val = best_model.predict(X_val_scaled)
    met_val = {
        "MSE": mean_squared_error(y_val, y_pred_val),
        "RMSE": np.sqrt(mean_squared_error(y_val, y_pred_val)),
        "MAE": mean_absolute_error(y_val, y_pred_val),
        "R2": r2_score(y_val, y_pred_val)
    }

    # Computing regression & evaluating accuracy metrics on test set
    y_pred_test = best_model.predict(X_test_scaled)
    met_test = {
        "MSE": mean_squared_error(y_test, y_pred_test),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "MAE": mean_absolute_error(y_test, y_pred_test),
        "R2": r2_score(y_test, y_pred_test)
    }

    return met_val, met_test, best_model

In [15]:
# Defining value for parameters to compute the search on
# param_grid = {
#     'n_estimators': [100, 200, 500],
#     'max_depth': [None, 5, 10, 20],
#     'min_samples_split': [2, 5],
#     'bootstrap': [True, False],
#     'oob_score': [True, False],
# }

In [26]:
# Best parameters (evaluated via grid search)
p_ecg_a = {'bootstrap': True, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200, 'oob_score': True}
p_ecg_v = {'bootstrap': True, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 500, 'oob_score': True}
p_eda_a = {'bootstrap': True, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100, 'oob_score': True}
p_eda_v = {'bootstrap': True, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 500, 'oob_score': True}

# Training models
val_ecg_a, test_ecg_a, model_ecg_a = train_and_evaluate(df[ecg_features], y_arousal, p_ecg_a)
val_ecg_v, test_ecg_v, model_ecg_v = train_and_evaluate(df[ecg_features], y_valence, p_ecg_v)
val_eda_a, test_eda_a, model_eda_a = train_and_evaluate(df[eda_features], y_arousal, p_eda_a)
val_eda_v, test_eda_v, model_eda_v = train_and_evaluate(df[eda_features], y_valence, p_eda_v)

**Graphic Display**

In [27]:
metrics_table = [
    {"Phase": "Validation", "Signal": "ECG", "Target": "Arousal", **val_ecg_a},
    {"Phase": "Test",       "Signal": "ECG", "Target": "Arousal", **test_ecg_a},
    {"Phase": "Validation", "Signal": "ECG", "Target": "Valence", **val_ecg_v},
    {"Phase": "Test",       "Signal": "ECG", "Target": "Valence", **test_ecg_v},
    {"Phase": "Validation", "Signal": "EDA", "Target": "Arousal", **val_eda_a},
    {"Phase": "Test",       "Signal": "EDA", "Target": "Arousal", **test_eda_a},
    {"Phase": "Validation", "Signal": "EDA", "Target": "Valence", **val_eda_v},
    {"Phase": "Test",       "Signal": "EDA", "Target": "Valence", **test_eda_v}
]

df_metrics = pd.DataFrame(metrics_table)
df_metrics.set_index(["Signal", "Target", "Phase"], inplace=True)
df_metrics = df_metrics.round(4)

display(df_metrics)

MSE    RMSE     MAE      R2
Signal Target  Phase                                     
ECG    Arousal Validation  1.4592  1.2080  0.9792  0.0229
               Test        2.0658  1.4373  1.1401 -0.0863
       Valence Validation  3.0137  1.7360  1.4972 -0.0249
               Test        4.2756  2.0678  1.7874 -0.0228
EDA    Arousal Validation  1.5244  1.2347  1.0089 -0.0208
               Test        1.8909  1.3751  1.1204  0.0057
       Valence Validation  3.1738  1.7815  1.5042 -0.0793
               Test        4.9737  2.2302  1.9109 -0.1898

In [28]:
# Saving models for Late Fusion
import joblib

joblib.dump(model_ecg_a, 'rf_amigos_ecg_arousal.pkl')
joblib.dump(model_ecg_v, 'rf_amigos_ecg_valence.pkl')
joblib.dump(model_eda_a, 'rf_amigos_eda_arousal.pkl')
joblib.dump(model_eda_v, 'rf_amigos_eda_valence.pkl')

['rf_amigos_eda_valence.pkl']